In [1]:
from google.colab import files
import pandas as pd

uploaded = files.upload()

Saving Merged_Disasters_Dataset.csv to Merged_Disasters_Dataset.csv


In [52]:
df = pd.read_csv('Merged_Disasters_Dataset.csv',low_memory=False)


In [53]:
import pandas as pd


missing_counts = df.isna().sum()
print("Missing values per column:")
print(missing_counts)

 #Percentage of missing values per column
missing_percent = (df.isna().sum() / len(df)) * 100
print("\nPercentage of missing values per column:")
print(missing_percent)


missing_summary = pd.DataFrame({
    'missing_count': missing_counts,
    'missing_percent': missing_percent
}).sort_values(by='missing_percent', ascending=False)

print("\nColumns with missing values:")
print(missing_summary[missing_summary['missing_count'] > 0])


Missing values per column:
Entity                                                       0
Code                                                       849
Year                                                         0
GDP, PPP (constant 2021 international $)                  2919
Share of the population using improved water sources     12101
GDP per capita, PPP (constant 2021 international $)       2919
Urban population                                          1369
Rural population                                          1369
Human Development Index                                   6656
indicator                                                 8135
estimate                                                  8135
Region                                                   14737
Disaster Group                                           17365
Disaster Type                                            17365
Total Deaths                                             68347
No. Injured                 

In [54]:
import pandas as pd
import numpy as np

print(f"Initial rows: {len(df)}")
print(f"Initial missing values: {df.isnull().sum().sum()}\n")

#Drop columns with >80% missing values
cols_to_drop = [
    'Total Damage (\'000 US$)_tech',
    'No. Affected_tech',
    'No. Homeless'
]
df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])
print(f" Dropped {len([col for col in cols_to_drop if col in df.columns])} columns\n")

# Handle disaster impact metrics (missing = 0, no disaster occurred)

disaster_impact_cols = [
    'Total Deaths',
    'No. Injured',
    'No. Affected',
    'Total Affected',
    'Total Damage (\'000 US$)',
    'Total Deaths_tech',
    'Total Affected_tech'
]

for col in disaster_impact_cols:
    if col in df.columns:
        df[col].fillna(0, inplace=True)
print(f" Filled {len([col for col in disaster_impact_cols if col in df.columns])} disaster impact columns\n")

# STEP 3: Handle disaster categorical columns (missing = 'No Disaster')

print("STEP 3: Filling disaster categorical columns...")
disaster_categorical = [
    'Disaster Type',
    'Disaster Group',
    'Disaster Type_tech',
    'Disaster Group_tech'
]

for col in disaster_categorical:
    if col in df.columns:
        df[col].fillna('No Disaster', inplace=True)
print(f" Filled {len([col for col in disaster_categorical if col in df.columns])} disaster categorical columns\n")

if 'Code' in df.columns:
    df['Code'].fillna('UNKNOWN', inplace=True)
print(" Code column filled\n")


df = df.sort_values(['Entity', 'Year']).reset_index(drop=True)

#Handle Region column

if 'Region' in df.columns:
    df['Region'] = df.groupby('Entity')['Region'].transform(
        lambda x: x.fillna(method='ffill').fillna(method='bfill')
    )

    df['Region'].fillna('Unknown', inplace=True)
print(" Region column filled\n")

if 'indicator' in df.columns:
    if df['indicator'].dtype == 'object':
        df['indicator'].fillna('Unknown', inplace=True)
    else:
        df['indicator'].fillna(df['indicator'].median(), inplace=True)

if 'estimate' in df.columns:
    if df['estimate'].dtype == 'object':
        df['estimate'].fillna('Unknown', inplace=True)
    else:
        df['estimate'].fillna(df['estimate'].median(), inplace=True)


# Handle economic indicators using INTERPOLATION

economic_cols = [
    'GDP, PPP (constant 2021 international $)',
    'GDP per capita, PPP (constant 2021 international $)',
    'Urban population',
    'Rural population',
    'Human Development Index',
    'Share of the population using improved water sources'
]

for col in economic_cols:
    if col in df.columns:

        df[col] = df.groupby('Entity')[col].transform(
            lambda x: x.interpolate(method='linear', limit_direction='both')
        )


        df[col].fillna(df[col].median(), inplace=True)

print(f" Interpolated {len([col for col in economic_cols if col in df.columns])} economic columns\n")

numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)

categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        mode_value = df[col].mode()
        if len(mode_value) > 0:
            df[col].fillna(mode_value[0], inplace=True)
        else:
            df[col].fillna('Unknown', inplace=True)

print(" All remaining missing values filled\n")


remaining_missing = df.isnull().sum()
total_missing = remaining_missing.sum()

if total_missing == 0:
    print("\n SUCCESS! NO MISSING VALUES REMAINING! \n")
else:
    print(f"\n WARNING: {total_missing} missing values still remain:\n")
    print(remaining_missing[remaining_missing > 0])

print(f"Total rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")
print(f"Rows with ANY missing values: {df.isnull().any(axis=1).sum()}")
print(f"Total missing values: {total_missing}")

print("\nDataframe Info:")
print(df.info())


Initial rows: 263777
Initial missing values: 1606020

 Dropped 0 columns

 Filled 7 disaster impact columns

STEP 3: Filling disaster categorical columns...


/tmp/ipython-input-597151263.py:30: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(0, inplace=True)
/tmp/ipython-input-597151263.py:45: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({c

 Filled 4 disaster categorical columns

 Code column filled



/tmp/ipython-input-597151263.py:62: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Region'].fillna('Unknown', inplace=True)
/tmp/ipython-input-597151263.py:67: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 

 Region column filled



/tmp/ipython-input-597151263.py:97: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)
/tmp/ipython-input-597151263.py:97: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try usin

 Interpolated 6 economic columns

 All remaining missing values filled


 SUCCESS! NO MISSING VALUES REMAINING! 

Total rows: 263777
Total columns: 23
Rows with ANY missing values: 0
Total missing values: 0

Dataframe Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 263777 entries, 0 to 263776
Data columns (total 23 columns):
 #   Column                                                Non-Null Count   Dtype  
---  ------                                                --------------   -----  
 0   Entity                                                263777 non-null  object 
 1   Code                                                  263777 non-null  object 
 2   Year                                                  263777 non-null  int64  
 3   GDP, PPP (constant 2021 international $)              263777 non-null  float64
 4   Share of the population using improved water sources  263777 non-null  float64
 5   GDP per capita, PPP (constant 2021 international $)   263777 non-null  f

In [55]:

exact_duplicates = df.duplicated().sum()
print(f"   Total exact duplicate rows: {exact_duplicates}")

if exact_duplicates > 0:
    print("\n   Sample of duplicate rows:")
    print(df[df.duplicated(keep=False)].head(10))


key_columns = ['Entity', 'Year']

key_duplicates = df.duplicated(subset=key_columns).sum()
print(f"   Duplicates based on {key_columns}: {key_duplicates}")

if key_duplicates > 0:
    print("\n   Sample of key-based duplicates:")
    duplicate_samples = df[df.duplicated(subset=key_columns, keep=False)].sort_values(key_columns)
    print(duplicate_samples.head(20))

    print("\n   Entities with duplicate years:")
    dup_entities = duplicate_samples.groupby('Entity')['Year'].apply(lambda x: x.value_counts())
    print(dup_entities.head(20))

   Total exact duplicate rows: 6044

   Sample of duplicate rows:
       Entity Code  Year  GDP, PPP (constant 2021 international $)  \
2498  Algeria  DZA  2003                              4.129825e+11   
2499  Algeria  DZA  2003                              4.129825e+11   
2500  Algeria  DZA  2003                              4.129825e+11   
2501  Algeria  DZA  2003                              4.129825e+11   
2502  Algeria  DZA  2003                              4.129825e+11   
2503  Algeria  DZA  2003                              4.129825e+11   
2504  Algeria  DZA  2003                              4.129825e+11   
2505  Algeria  DZA  2003                              4.129825e+11   
2506  Algeria  DZA  2003                              4.129825e+11   
2507  Algeria  DZA  2003                              4.129825e+11   

      Share of the population using improved water sources  \
2498                                           96.10438      
2499                                   

In [56]:

initial_rows = len(df)
print(f"\nInitial dataset:")
print(f"   Total rows: {initial_rows}")

exact_duplicates = df.duplicated().sum()
print(f"   Exact duplicates found: {exact_duplicates}")

if exact_duplicates > 0:
    print(f"\n Sample of exact duplicates (will be removed):")
    print(df[df.duplicated(keep=False)].head(10))

df = df.drop_duplicates(keep='first')


final_rows = len(df)
removed_rows = initial_rows - final_rows

print(f"\n" + "="*60)
print(" EXACT DUPLICATES REMOVED!")
print("="*60)
print(f"\nResults:")
print(f"   Rows before: {initial_rows}")
print(f"   Rows after: {final_rows}")
print(f"   Removed: {removed_rows} exact duplicate rows")

remaining_exact_duplicates = df.duplicated().sum()
print(f"   Exact duplicates remaining: {remaining_exact_duplicates}")


entity_year_duplicates = df.duplicated(subset=['Entity', 'Year']).sum()
print(f"\n Entity-Year duplicates remaining: {entity_year_duplicates}")
print(f"   (This is GOOD - different disasters for same country-year)")

print(f"\n Example: Afghanistan 2000 (different disasters kept):")
sample = df[(df['Entity'] == 'Afghanistan') & (df['Year'] == 2000)][
    ['Entity', 'Year', 'Disaster Type', 'Total Deaths', 'Total Affected']
]
print(sample.head(10))

df = df.reset_index(drop=True)



Initial dataset:
   Total rows: 263777
   Exact duplicates found: 6044

 Sample of exact duplicates (will be removed):
       Entity Code  Year  GDP, PPP (constant 2021 international $)  \
2498  Algeria  DZA  2003                              4.129825e+11   
2499  Algeria  DZA  2003                              4.129825e+11   
2500  Algeria  DZA  2003                              4.129825e+11   
2501  Algeria  DZA  2003                              4.129825e+11   
2502  Algeria  DZA  2003                              4.129825e+11   
2503  Algeria  DZA  2003                              4.129825e+11   
2504  Algeria  DZA  2003                              4.129825e+11   
2505  Algeria  DZA  2003                              4.129825e+11   
2506  Algeria  DZA  2003                              4.129825e+11   
2507  Algeria  DZA  2003                              4.129825e+11   

      Share of the population using improved water sources  \
2498                                           

In [57]:

print(f"\nCurrent dataset: {df.shape[0]} rows × {df.shape[1]} columns")


print(f"Missing values: {df.isnull().sum().sum()}")

print(f"Exact duplicates: {df.duplicated().sum()}")

print("\n" + "-"*60)
print(" Data Type Corrections")
print("-"*60)


if 'Year' in df.columns:
    df['Year'] = df['Year'].astype(int)
    print(" Year converted to integer")

numeric_cols = [
    'GDP, PPP (constant 2021 international $)',
    'GDP per capita, PPP (constant 2021 international $)',
    'Total Deaths', 'No. Injured', 'No. Affected', 'Total Affected',
    'Total Damage (\'000 US$)', 'Total Deaths_tech', 'Total Affected_tech',
    'Urban population', 'Rural population', 'Human Development Index',
    'Share of the population using improved water sources'
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print("\n" + "-"*60)
print("STEP 4: Check for Negative Values")
print("-"*60)

negative_check_cols = [
    'Total Deaths', 'No. Injured', 'No. Affected', 'Total Affected',
    'Total Damage (\'000 US$)', 'Total Deaths_tech', 'Total Affected_tech',
    'GDP, PPP (constant 2021 international $)',
    'GDP per capita, PPP (constant 2021 international $)',
    'Urban population', 'Rural population'
]

negative_found = False
for col in negative_check_cols:
    if col in df.columns:
        negative_count = (df[col] < 0).sum()
        if negative_count > 0:
            print(f"   {col}: {negative_count} negative values found")
            # Replace with 0
            df.loc[df[col] < 0, col] = 0
            negative_found = True

if not negative_found:
    print(" No negative values found")
else:
    print(" Negative values replaced with 0")


if 'Year' in df.columns:
    year_min, year_max = df['Year'].min(), df['Year'].max()
    print(f"  Year range: {year_min} to {year_max}")


    current_year = 2025
    invalid_years = df[(df['Year'] < 1900) | (df['Year'] > current_year)]
    if len(invalid_years) > 0:
        df = df[(df['Year'] >= 1900) & (df['Year'] <= current_year)]
        print(f"   Removed {len(invalid_years)} rows with invalid years")
    else:
        print(f"   All years valid (1900-{current_year})")


if 'Human Development Index' in df.columns:
    invalid_hdi = df[(df['Human Development Index'] < 0) | (df['Human Development Index'] > 1)]
    if len(invalid_hdi) > 0:
        print(f"   Found {len(invalid_hdi)} rows with invalid HDI")
        df = df[(df['Human Development Index'] >= 0) & (df['Human Development Index'] <= 1)]
    else:
        print(f"   HDI values valid (0-1)")



text_cols = ['Entity', 'Code', 'Region', 'Disaster Type', 'Disaster Group',
             'Disaster Type_tech', 'Disaster Group_tech', 'indicator', 'estimate']

for col in text_cols:
    if col in df.columns:

        df[col] = df[col].astype(str).str.strip()






def detect_outliers_iqr(data, column):
    """Detect outliers using IQR method"""
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = data[(data[column] < lower) | (data[column] > upper)]
    return len(outliers), lower, upper

outlier_check_cols = ['Total Deaths', 'Total Affected', 'GDP per capita, PPP (constant 2021 international $)']

for col in outlier_check_cols:
    if col in df.columns and df[col].notna().sum() > 0:
        n_outliers, lower, upper = detect_outliers_iqr(df, col)
        print(f"\n  {col}:")
        print(f"    Outliers: {n_outliers} ({n_outliers/len(df)*100:.2f}%)")
        print(f"    Range: [{lower:.0f}, {upper:.0f}]")

print("\n  Note: Outliers NOT removed (legitimate extreme disaster events)")



print(f"\n Dataset Summary:")
print(f"   Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"   Date range: {df['Year'].min()} to {df['Year'].max()}")
print(f"   Unique entities: {df['Entity'].nunique()}")
print(f"   Missing values: {df.isnull().sum().sum()}")
print(f"   Exact duplicates: {df.duplicated().sum()}")


# Display data info
print("\n Data Types:")
print(df.dtypes)


Current dataset: 257733 rows × 23 columns
Missing values: 0
Exact duplicates: 0

------------------------------------------------------------
 Data Type Corrections
------------------------------------------------------------
 Year converted to integer

------------------------------------------------------------
STEP 4: Check for Negative Values
------------------------------------------------------------
 No negative values found
  Year range: 1990 to 2024
   All years valid (1900-2025)
   HDI values valid (0-1)

  Total Deaths:
    Outliers: 32472 (12.60%)
    Range: [-52, 88]

  Total Affected:
    Outliers: 48584 (18.85%)
    Range: [-195013, 325075]

  GDP per capita, PPP (constant 2021 international $):
    Outliers: 18795 (7.29%)
    Range: [-5871, 21839]

  Note: Outliers NOT removed (legitimate extreme disaster events)

 Dataset Summary:
   Shape: 257733 rows × 23 columns
   Date range: 1990 to 2024
   Unique entities: 304
   Missing values: 0
   Exact duplicates: 0

 Data T

In [58]:



print("\n" + "-"*60)
print("1. Creating TOTAL POPULATION")
print("-"*60)

if 'Urban population' in df.columns and 'Rural population' in df.columns:
    df['Total Population'] = df['Urban population'] + df['Rural population']
    print(f"Total Population created")
    print(f"   Range: {df['Total Population'].min():,.0f} to {df['Total Population'].max():,.0f}")
    print(f"   Mean: {df['Total Population'].mean():,.0f}")
else:
    print("Warning: Urban/Rural population columns not found")

print("\n" + "-"*60)
print("2. Creating SEVERITY WEIGHT (S) for Disaster Types")
print("-"*60)

# Define severity weights based on disaster type (0-1 scale)
severity_weights = {
    # Natural Disasters, High Severity
    'Earthquake': 0.95,
    'Tsunami': 0.90,
    'Tropical cyclone': 0.85,
    'Flood': 0.70,
    'Landslide': 0.75,
    'Volcanic activity': 0.80,
    'Extreme temperature': 0.65,
    'Drought': 0.60,
    'Wildfire': 0.55,
    'Storm': 0.70,

    # Epidemics
    'Epidemic': 0.75,
    'Pandemic': 0.95,

    # Other
    'Mass movement (dry)': 0.50,
    'Insect infestation': 0.30,
    'Animal accident': 0.40,
    'Fog': 0.20,
    'Glacial lake outburst': 0.65,

    # Technological Disasters
    'Air': 0.70,
    'Road': 0.50,
    'Rail': 0.60,
    'Water': 0.55,
    'Collapse (Miscellaneous)': 0.65,
    'Explosion': 0.75,
    'Fire': 0.60,
    'Gas leak': 0.55,
    'Poisoning': 0.50,
    'Radiation': 0.85,
    'Oil spill': 0.45,

    # Default
    'No Disaster': 0.0,
    'Unknown': 0.50
}

# Apply severity weights to natural disasters
df['Severity_Weight_Natural'] = df['Disaster Type'].map(severity_weights)
df['Severity_Weight_Natural'].fillna(0.5, inplace=True)

# Apply severity weights to technological disasters
df['Severity_Weight_Tech'] = df['Disaster Type_tech'].map(severity_weights)
df['Severity_Weight_Tech'].fillna(0.5, inplace=True)

# Combined severity weight (maximum of both)
df['Severity_Weight'] = df[['Severity_Weight_Natural', 'Severity_Weight_Tech']].max(axis=1)

print(f"Severity weights created")
print(f"\nTop 10 most severe disaster types:")
severity_summary = df.groupby('Disaster Type')['Severity_Weight_Natural'].first().sort_values(ascending=False).head(10)
print(severity_summary)


print("\n" + "-"*60)
print("3. Creating DISASTER INTENSITY SCORES")
print("-"*60)

# Normalize deaths and affected population
if 'Total Population' in df.columns:
    # Death Rate (per 100,000 population)
    df['Death_Rate'] = (df['Total Deaths'] / df['Total Population'] * 100000).fillna(0)

    # Affected Rate (per 100,000 population)
    df['Affected_Rate'] = (df['Total Affected'] / df['Total Population'] * 100000).fillna(0)

    print(f"Death Rate created (per 100k population)")
    print(f"Affected Rate created (per 100k population)")

if 'Total Damage (\'000 US$)' in df.columns and 'GDP, PPP (constant 2021 international $)' in df.columns:
    df['Damage_GDP_Ratio'] = (df['Total Damage (\'000 US$)'] * 1000 / df['GDP, PPP (constant 2021 international $)'] * 100).fillna(0)
    df['Damage_GDP_Ratio'] = df['Damage_GDP_Ratio'].clip(0, 100)
    print(f"Economic Damage Ratio created (% of GDP)")

df['Disaster_Intensity'] = (
    (df['Severity_Weight'] * 25) +
    (np.minimum(df['Death_Rate'] / 10, 25)) +
    (np.minimum(df['Affected_Rate'] / 100, 25)) +
    (df['Damage_GDP_Ratio'] * 0.25)
).fillna(0)

print(f"\nDisaster Intensity Score created (0-100 scale)")
print(f"   Mean: {df['Disaster_Intensity'].mean():.2f}")
print(f"   Max: {df['Disaster_Intensity'].max():.2f}")


df['Intensity_Category'] = pd.cut(
    df['Disaster_Intensity'],
    bins=[-1, 0, 10, 25, 50, 100],
    labels=['No Disaster', 'Low', 'Moderate', 'High', 'Catastrophic']
)

print(f"\nIntensity Category created")
print("\nDistribution:")
print(df['Intensity_Category'].value_counts().sort_index())

print("\n" + "-"*60)
print("4. Creating ADAPTIVE CAPACITY (A) Index")
print("-"*60)

if 'GDP per capita, PPP (constant 2021 international $)' in df.columns:
    gdp_max = df['GDP per capita, PPP (constant 2021 international $)'].max()
    df['GDP_Capacity'] = (df['GDP per capita, PPP (constant 2021 international $)'] / gdp_max).fillna(0)
    print(f"Economic Capacity normalized")

# HDI is already 0-1 scale
if 'Human Development Index' in df.columns:
    df['HDI_Capacity'] = df['Human Development Index'].fillna(df['Human Development Index'].median())
    print(f"Human Development Capacity normalized")

# Urbanization capacity (0-1 scale)
if 'Urban population' in df.columns and 'Total Population' in df.columns:
    df['Urbanization_Capacity'] = (df['Urban population'] / df['Total Population']).fillna(0.5)
    print(f"Urbanization Capacity normalized")

# Water access capacity (0-1 scale)
if 'Share of the population using improved water sources' in df.columns:
    df['Water_Capacity'] = (df['Share of the population using improved water sources'] / 100).fillna(0.5)
    print(f"Water Access Capacity normalized")

# COMPOSITE ADAPTIVE CAPACITY INDEX (0-1 scale)
df['Adaptive_Capacity'] = (
    df['GDP_Capacity'] * 0.35 +
    df['HDI_Capacity'] * 0.30 +
    df['Urbanization_Capacity'] * 0.20 +
    df['Water_Capacity'] * 0.15
).fillna(0.5)

print(f"\nAdaptive Capacity Index created (0-1 scale)")
print(f"   Mean: {df['Adaptive_Capacity'].mean():.3f}")
print(f"   Range: {df['Adaptive_Capacity'].min():.3f} to {df['Adaptive_Capacity'].max():.3f}")

# Categorize adaptive capacity
df['Adaptive_Capacity_Level'] = pd.cut(
    df['Adaptive_Capacity'],
    bins=[0, 0.3, 0.5, 0.7, 1.0],
    labels=['Low', 'Moderate', 'High', 'Very High']
)


print("\nDistribution:")
print(df['Adaptive_Capacity_Level'].value_counts().sort_index())


print("\n" + "-"*60)
print("5. Creating T_RECOVERY (GDP Recovery Time)")
print("-"*60)

# Sort by Entity and Year for time-series calculations
df = df.sort_values(['Entity', 'Year']).reset_index(drop=True)

# Calculate GDP growth rate
df['GDP_Growth_Rate'] = df.groupby('Entity')['GDP, PPP (constant 2021 international $)'].pct_change() * 100

# Fill GDP Growth Rate missing values
print("Filling GDP Growth Rate missing values...")
df['GDP_Growth_Rate'] = df.groupby('Entity')['GDP_Growth_Rate'].transform(
    lambda x: x.fillna(method='ffill').fillna(method='bfill').fillna(x.mean())
)
df['GDP_Growth_Rate'].fillna(df['GDP_Growth_Rate'].median(), inplace=True)

print(f"GDP Growth Rate missing values: {df['GDP_Growth_Rate'].isnull().sum()}")

# Calculate GDP loss from disasters (as % of GDP)
df['GDP_Loss_Percent'] = df['Damage_GDP_Ratio'].copy()
df['GDP_Loss_Percent'].fillna(0, inplace=True)

# Calculate average growth rate per entity (excluding disaster years)
avg_growth_dict = df[df['Disaster_Intensity'] < 10].groupby('Entity')['GDP_Growth_Rate'].mean().to_dict()
df['Avg_Growth_Rate'] = df['Entity'].map(avg_growth_dict)

# Fill missing with entity's overall mean
entity_mean = df.groupby('Entity')['GDP_Growth_Rate'].mean().to_dict()
df['Avg_Growth_Rate'] = df['Avg_Growth_Rate'].fillna(df['Entity'].map(entity_mean))

# Fill remaining with global mean
df['Avg_Growth_Rate'].fillna(df['GDP_Growth_Rate'].mean(), inplace=True)

print(f"Avg Growth Rate missing values: {df['Avg_Growth_Rate'].isnull().sum()}")

# Calculate T_recovery (in years)
df['T_Recovery_Years'] = 0.0

mask = (
    (df['GDP_Loss_Percent'] > 0) &
    (df['Avg_Growth_Rate'] > 0) &
    (df['Adaptive_Capacity'] > 0)
)

df.loc[mask, 'T_Recovery_Years'] = (
    df.loc[mask, 'GDP_Loss_Percent'] /
    (df.loc[mask, 'Avg_Growth_Rate'] * df.loc[mask, 'Adaptive_Capacity'])
)

# Handle infinities and cap at 20 years
df['T_Recovery_Years'] = df['T_Recovery_Years'].replace([np.inf, -np.inf], 20)
df['T_Recovery_Years'] = df['T_Recovery_Years'].clip(0, 20)

print(f"T_Recovery created")
print(f"   Mean recovery time: {df[df['T_Recovery_Years'] > 0]['T_Recovery_Years'].mean():.2f} years")
print(f"   Max recovery time: {df['T_Recovery_Years'].max():.2f} years")

# Recovery category
df['Recovery_Speed'] = pd.cut(
    df['T_Recovery_Years'],
    bins=[-1, 0, 1, 3, 5, 20],
    labels=['No Recovery Needed', 'Fast (<1 yr)', 'Moderate (1-3 yrs)', 'Slow (3-5 yrs)', 'Very Slow (>5 yrs)']
)
df['Recovery_Speed'].fillna('No Recovery Needed', inplace=True)

print(f"\nRecovery Speed Category created")
print("\nDistribution:")
print(df['Recovery_Speed'].value_counts())


remaining_missing = df.isnull().sum().sum()
print(f"\nRemaining missing values: {remaining_missing}")

if remaining_missing > 0:
    print("\nFilling all remaining missing values...")

    # Numeric: fill with 0
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if df[col].isnull().sum() > 0:
            df[col].fillna(0, inplace=True)
            print(f"   Filled {col}")

    # Categorical: fill with 'Unknown'
    cat_cols = df.select_dtypes(include=['object', 'category']).columns
    for col in cat_cols:
        if df[col].isnull().sum() > 0:
            df[col].fillna('Unknown', inplace=True)
            print(f"   Filled {col}")

print(f"\nFinal missing values: {df.isnull().sum().sum()}")



new_columns = [
    'Total Population',
    'Severity_Weight_Natural',
    'Severity_Weight_Tech',
    'Severity_Weight',
    'Death_Rate',
    'Affected_Rate',
    'Damage_GDP_Ratio',
    'Disaster_Intensity',
    'Intensity_Category',
    'GDP_Capacity',
    'HDI_Capacity',
    'Urbanization_Capacity',
    'Water_Capacity',
    'Adaptive_Capacity',
    'Adaptive_Capacity_Level',
    'GDP_Growth_Rate',
    'GDP_Loss_Percent',
    'Avg_Growth_Rate',
    'T_Recovery_Years',
    'Recovery_Speed'
]

print(f"\nNew Columns Created: {len(new_columns)}")
print("\nColumn List:")
for i, col in enumerate(new_columns, 1):
    if col in df.columns:
        print(f"   {i}. {col}")

print(f"\nFinal Dataset Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Total missing values: {df.isnull().sum().sum()}")

# Display sample with new variables
print("\nSample of new variables:")
sample_cols = ['Entity', 'Year', 'Disaster Type', 'Severity_Weight',
               'Disaster_Intensity', 'Adaptive_Capacity', 'T_Recovery_Years']
print(df[sample_cols].head(10))




------------------------------------------------------------
1. Creating TOTAL POPULATION
------------------------------------------------------------
Total Population created
   Range: 8,798 to 8,140,502,993
   Mean: 757,782,722

------------------------------------------------------------
2. Creating SEVERITY WEIGHT (S) for Disaster Types
------------------------------------------------------------
Severity weights created

Top 10 most severe disaster types:
Disaster Type
Earthquake                     0.95
Volcanic activity              0.80
Epidemic                       0.75
Flood                          0.70
Storm                          0.70
Extreme temperature            0.65
Drought                        0.60
Wildfire                       0.55
Animal incident                0.50
Glacial lake outburst flood    0.50
Name: Severity_Weight_Natural, dtype: float64

------------------------------------------------------------
3. Creating DISASTER INTENSITY SCORES
--------------

/tmp/ipython-input-4089887715.py:62: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Severity_Weight_Natural'].fillna(0.5, inplace=True)
/tmp/ipython-input-4089887715.py:66: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)

Filling GDP Growth Rate missing values...


/tmp/ipython-input-4089887715.py:180: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  lambda x: x.fillna(method='ffill').fillna(method='bfill').fillna(x.mean())
/tmp/ipython-input-4089887715.py:182: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['GDP_Growth_Rate'].fillna(df['GDP_Growth_Rate'].median(), inplace=True)
/tmp/ipython-input-4089887715.py:188: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through c

GDP Growth Rate missing values: 0


/tmp/ipython-input-4089887715.py:199: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Avg_Growth_Rate'].fillna(df['GDP_Growth_Rate'].mean(), inplace=True)


Avg Growth Rate missing values: 0
T_Recovery created
   Mean recovery time: 0.08 years
   Max recovery time: 20.00 years

Recovery Speed Category created

Distribution:
Recovery_Speed
No Recovery Needed    163206
Fast (<1 yr)           93419
Moderate (1-3 yrs)       650
Very Slow (>5 yrs)       306
Slow (3-5 yrs)           152
Name: count, dtype: int64


/tmp/ipython-input-4089887715.py:231: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Recovery_Speed'].fillna('No Recovery Needed', inplace=True)



Remaining missing values: 0

Final missing values: 0

New Columns Created: 20

Column List:
   1. Total Population
   2. Severity_Weight_Natural
   3. Severity_Weight_Tech
   4. Severity_Weight
   5. Death_Rate
   6. Affected_Rate
   7. Damage_GDP_Ratio
   8. Disaster_Intensity
   9. Intensity_Category
   10. GDP_Capacity
   11. HDI_Capacity
   12. Urbanization_Capacity
   13. Water_Capacity
   14. Adaptive_Capacity
   15. Adaptive_Capacity_Level
   16. GDP_Growth_Rate
   17. GDP_Loss_Percent
   18. Avg_Growth_Rate
   19. T_Recovery_Years
   20. Recovery_Speed

Final Dataset Shape: 257733 rows x 43 columns
Total missing values: 0

Sample of new variables:
        Entity  Year Disaster Type  Severity_Weight  Disaster_Intensity  \
0  Afghanistan  2000      Epidemic             0.75           19.001859   
1  Afghanistan  2000      Epidemic             0.75           18.758495   
2  Afghanistan  2000      Epidemic             0.75           18.839119   
3  Afghanistan  2000      Epidemic 

In [59]:
df.isnull().sum()

,0
Entity,0
Code,0
Year,0
"GDP, PPP (constant 2021 international $)",0
Share of the population using improved water sources,0
"GDP per capita, PPP (constant 2021 international $)",0
Urban population,0
Rural population,0
Human Development Index,0
indicator,0


In [60]:
df.to_csv('Merged_Disasters_Cleaned_Dataset.csv', index=False)


files.download('Merged_Disasters_Cleaned_Dataset.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>